# Computer Vision — Hands-on Lab 05
## Building a CNN from scratch: Cat vs Dog

---

In this lab you will build a **Convolutional Neural Network** in PyTorch that looks at
a photograph and answers one question: **cat or dog?**

### What a CNN does, in one paragraph

A CNN looks at an image through small windows. It takes a little grid of numbers — a
**filter**, usually 3x3 — slides it across the image, and at every position multiplies
and adds. Where the image matches the pattern in the filter, the result is large;
where it does not, the result is small. The output is a new image showing *where that
pattern was found*. A CNN stacks many of these, and the numbers inside the filters are
**learned from the data** rather than chosen by a person.

That is the whole idea. Everything else in this notebook is plumbing around it.

### The pipeline you are building

```
Image (any size, JPEG on disk)
  ↓  Transform / preprocessing
Tensor  (3 x 128 x 128)
  ↓  Conv → ReLU → Pool   (x3)
Feature maps  (64 x 16 x 16)
  ↓  Flatten
Feature vector  (16384 numbers)
  ↓  Linear classifier
One logit
  ↓  sigmoid
P(dog)
```

### What this lab is really about

The goal is **not** a high accuracy number. A small CNN trained on a couple of
thousand images for a few minutes will land somewhere around 75–85% — respectable,
not impressive, and beside the point.

The goal is that by the end you can point at any line of the model and say what it
does to the image flowing through it. The question to keep asking yourself is:

> **"What is the shape of the thing at this point, and why?"**

### How to work through it

Sections are short on purpose: read the explanation, run the cell, *look at the
output*, answer the question. There are **7 TODO exercises** marked like this:

> **TODO 3** — change the number of filters from 16 to 32.

Do them. They are the difference between reading a notebook and learning from one.

---

---
# 0. Setup

Four libraries:

- **PyTorch (`torch`)** — tensors, layers, training.
- **torchvision** — image datasets and the transform pipeline.
- **Matplotlib** — looking at images, feature maps and training curves.
- **NumPy / PIL** — arrays and image files.

If `torch` is not installed, uncomment and run the install line below **once**. The
CPU build is enough for this notebook; if you have an NVIDIA GPU, use the selector at
<https://pytorch.org/get-started/locally/> to get the CUDA build instead.

In [ ]:
# Run once if needed, then comment it out again:
# %pip install torch torchvision matplotlib pillow

import random
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image

print("torch :", torch.__version__)
print("numpy :", np.__version__)

## 0.1 Reproducibility and device

**Seeds.** A CNN starts with *random* filters, the data is shuffled in a random order,
and augmentation flips images at random. Fixing the seeds means that when you rerun
the notebook you get the same numbers — so any difference you see later was caused by
the change *you* made, not by luck.

**Device.** `cuda` if you have an NVIDIA GPU, otherwise `cpu`. Everything here is
small enough to train on a CPU in a few minutes; a GPU just makes it faster. Whatever
it prints, the rest of the notebook works.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

---
## 0.2 Where the data comes from

We use the classic **Cats vs Dogs** dataset: 25,000 photos, originally from Microsoft
Research and the Kaggle "Dogs vs. Cats" competition. Real photographs, real mess —
different cameras, sizes, lighting and poses, plus a handful of files that are not
even valid images.

**Getting it** (you only need one of these):

1. **Microsoft's official download** (786 MB zip):
   <https://www.microsoft.com/en-us/download/details.aspx?id=54765>
   Unzip it and use the `PetImages` folder, which contains `Cat/` and `Dog/`.
2. **Kaggle**: <https://www.kaggle.com/datasets/tongpython/cat-and-dog>
3. **You may already have it** from an earlier lab.

Whatever you downloaded, you need one folder containing a `Cat/` folder and a `Dog/`
folder. Set `SOURCE_DIR` to point at it.

In [ ]:
# Change this line if your images are somewhere else.
SOURCE_DIR = Path("data")

if not SOURCE_DIR.exists():
    SOURCE_DIR = Path("..") / "AI_builders_CV" / "data"      # a second place to look

CAT_DIR = SOURCE_DIR / "Cat"
DOG_DIR = SOURCE_DIR / "Dog"

if not CAT_DIR.exists() or not DOG_DIR.exists():
    raise FileNotFoundError(
        f"Expected a Cat/ and a Dog/ folder inside {SOURCE_DIR.resolve()}.\n"
        "Download the dataset (links above) and set SOURCE_DIR to the folder that contains them."
    )

cat_files = sorted(CAT_DIR.glob("*.jpg"))
dog_files = sorted(DOG_DIR.glob("*.jpg"))

print("Source folder :", SOURCE_DIR.resolve())
print("Cat images    :", len(cat_files))
print("Dog images    :", len(dog_files))

## 0.3 Building a small train / val / test split

The full dataset is 25,000 images. We deliberately use only a small part of it: a
couple of thousand images is enough to watch a CNN learn, and it keeps one training
run down to a few minutes on a laptop.

We are going to build this folder structure, which is exactly what torchvision
expects:

```
dataset/
├── train/
│   ├── cats/     1000 images
│   └── dogs/     1000 images
├── val/
│   ├── cats/      200 images
│   └── dogs/      200 images
└── test/
    ├── cats/      200 images
    └── dogs/      200 images
```

Two small helper functions do the work:

- `good_images(...)` keeps only files that PIL can actually open. This dataset really
  does contain a few broken JPEGs, and it is much nicer to find them now than to have
  training crash halfway through.
- `copy_images(...)` copies a list of files into a folder.

Then we shuffle (with the seed, so everyone gets the same split), slice the list into
three parts, and copy. **The three splits never share an image** — that is the whole
point of splitting, and Section 3 explains why.

In [ ]:
def good_images(files, how_many):
    """Return the first `how_many` files that PIL can open without an error."""
    good = []
    for file in files:
        if len(good) == how_many:
            break
        try:
            Image.open(file).convert("RGB")     # if this works, the file is fine
            good.append(file)
        except Exception:
            print("  skipping unreadable file:", file.name)
    return good


def copy_images(files, folder):
    """Copy a list of files into `folder`, creating it if needed."""
    folder.mkdir(parents=True, exist_ok=True)
    for file in files:
        shutil.copy(file, folder / file.name)

In [ ]:
DATA_ROOT = Path("dataset")

N_TRAIN = 1000      # images per class
N_VAL = 200
N_TEST = 200
N_TOTAL = N_TRAIN + N_VAL + N_TEST

if DATA_ROOT.exists():
    print(f"{DATA_ROOT}/ already exists - skipping. Delete the folder if you want to rebuild it.")
else:
    for class_name, files in [("cats", cat_files), ("dogs", dog_files)]:
        print("preparing", class_name)

        random.Random(SEED).shuffle(files)          # same shuffle for everyone
        selected = good_images(files, N_TOTAL)

        train_part = selected[:N_TRAIN]
        val_part = selected[N_TRAIN:N_TRAIN + N_VAL]
        test_part = selected[N_TRAIN + N_VAL:]

        copy_images(train_part, DATA_ROOT / "train" / class_name)
        copy_images(val_part, DATA_ROOT / "val" / class_name)
        copy_images(test_part, DATA_ROOT / "test" / class_name)

    print("Done.")

In [ ]:
# What did we end up with?
for split in ["train", "val", "test"]:
    n_cats = len(list((DATA_ROOT / split / "cats").glob("*.jpg")))
    n_dogs = len(list((DATA_ROOT / split / "dogs").glob("*.jpg")))
    print(f"{split:6s} cats: {n_cats:5d}   dogs: {n_dogs:5d}   total: {n_cats + n_dogs:5d}")

---
# 1. Problem Definition

The task, stated precisely:

> Given one photograph, decide: **cat** or **dog**.

This is **binary image classification**. Two words, both worth unpacking:

- **Classification** — the output is a category, not a number and not a location. We
  are not asked *where* the animal is in the picture, only *what* it is.
- **Binary** — exactly two categories, and every image is one or the other. This is
  what will let us get away with a single output number later on.

Let us look at what we are actually dealing with. First, a small helper for drawing
images side by side — we will use it throughout the notebook.

In [ ]:
def show_images(images, titles, columns=4, size=3):
    """Draw a row (or grid) of images with titles above them."""
    rows = int(np.ceil(len(images) / columns))
    plt.figure(figsize=(size * columns, size * rows))

    for i in range(len(images)):
        plt.subplot(rows, columns, i + 1)
        plt.imshow(images[i], cmap="gray")     # cmap is ignored for colour images
        plt.title(titles[i], fontsize=10)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# The image files we will work with from now on
train_cat_paths = sorted((DATA_ROOT / "train" / "cats").glob("*.jpg"))
train_dog_paths = sorted((DATA_ROOT / "train" / "dogs").glob("*.jpg"))

# Four cats and four dogs, straight from disk, untouched
images = []
titles = []

for path in train_cat_paths[:4]:
    image = Image.open(path).convert("RGB")
    images.append(image)
    titles.append(f"Cat  {image.size[0]}x{image.size[1]}")

for path in train_dog_paths[:4]:
    image = Image.open(path).convert("RGB")
    images.append(image)
    titles.append(f"Dog  {image.size[0]}x{image.size[1]}")

show_images(images, titles)

### Question to think about before writing any code

> **What information does a model need in order to tell these two classes apart?**

Try to be concrete. Some candidate answers, none of them good enough on its own:

- Ear shape? Cats have small triangular ears — but so do plenty of dogs.
- Snout length? Usually longer on a dog — but not on a pug.
- Colour? Useless. Both come in every colour.
- Body shape? Only when the whole body is visible, which often it is not.

Notice what all of these have in common: they are **small local patterns in a
particular arrangement**. That is exactly the kind of thing a convolution is good at
finding.

Notice also what we are *not* going to do: we are not going to write down any of
these rules. We will show the network 2,000 labelled examples and let it work out for
itself what is worth measuring.

---
# 2. Explore the Dataset

Before modelling anything, look at the data. Two questions:

1. **How many images do we have, and where?**
2. **How similar are they to each other?** (Spoiler: not very.)

In [ ]:
def count_images(split, class_name):
    """How many jpg files are in dataset/<split>/<class_name>/ ?"""
    return len(list((DATA_ROOT / split / class_name).glob("*.jpg")))


print("split    cats    dogs   total")
print("-" * 30)

for split in ["train", "val", "test"]:
    n_cats = count_images(split, "cats")
    n_dogs = count_images(split, "dogs")
    print(f"{split:8s}{n_cats:5d}{n_dogs:8d}{n_cats + n_dogs:8d}")

## 2.1 A grid of real examples

The title above each image is its **original size on disk**. Look at the range of
numbers, and at how different the photos are from each other.

In [ ]:
random.seed(SEED)

picked = random.sample(train_cat_paths, 6) + random.sample(train_dog_paths, 6)

images = []
titles = []
for i, path in enumerate(picked):
    image = Image.open(path).convert("RGB")
    name = "Cat" if i < 6 else "Dog"
    images.append(image)
    titles.append(f"{name}  {image.size[0]}x{image.size[1]}")

show_images(images, titles, columns=6, size=2.5)

## 2.2 How different are these images, really?

Let us measure the variation instead of just eyeballing it. We read the size of every
training image and look at the range.

In [ ]:
widths = []
heights = []

for path in train_cat_paths + train_dog_paths:
    image = Image.open(path)
    widths.append(image.size[0])
    heights.append(image.size[1])

widths = np.array(widths)
heights = np.array(heights)
ratios = widths / heights          # aspect ratio: wider than tall is > 1

print(f"width  : smallest {widths.min()}, largest {widths.max()}, average {widths.mean():.0f}")
print(f"height : smallest {heights.min()}, largest {heights.max()}, average {heights.mean():.0f}")
print(f"ratio  : smallest {ratios.min():.2f}, largest {ratios.max():.2f}")

plt.figure(figsize=(11, 3.5))

plt.subplot(1, 2, 1)
plt.hist(widths, bins=40, color="steelblue")
plt.title("Image widths (pixels)")

plt.subplot(1, 2, 2)
plt.hist(ratios, bins=40, color="indianred")
plt.axvline(1.0, color="black", linestyle="--", label="square")
plt.title("Aspect ratio (width / height)")
plt.legend()

plt.tight_layout()
plt.show()

### What that is telling you

Almost every image has a **different size**, and the shapes range from tall portraits
to wide landscapes. On top of that, and visible in the grid above:

- **Different backgrounds** — sofas, grass, carpets, arms holding the animal.
- **Different poses** — sitting, lying, jumping, seen from behind.
- **Different scales** — a face filling the frame, or a small animal in a big room.
- **Different lighting** — flash, daylight, dim indoor shots.
- **Clutter** — a hand, a blanket, a second animal, text on the image.

This matters for two separate reasons:

1. **Practically:** we cannot stack images of different sizes into one batch, so
   something has to give. That is Section 5.
2. **Conceptually:** the network has to find features that survive all this
   variation. A rule like "the cat is in the middle of the picture" would fail on the
   very next photo.

---
# 3. Class Distribution

Now check the balance between the two classes. This is a quick check that people skip,
and it changes how you are allowed to read every accuracy number afterwards.

In [ ]:
plt.figure(figsize=(12, 3.2))

for position, split in enumerate(["train", "val", "test"]):
    n_cats = count_images(split, "cats")
    n_dogs = count_images(split, "dogs")

    plt.subplot(1, 3, position + 1)
    bars = plt.bar(["cats", "dogs"], [n_cats, n_dogs], color=["#6a9fb5", "#c98b6b"])
    plt.bar_label(bars)
    plt.title(f"{split}  (total {n_cats + n_dogs})")
    plt.ylim(0, 1100)

plt.tight_layout()
plt.show()

### Our dataset is balanced — 50 / 50 in every split

We built it that way on purpose, and it makes life simpler:

- **Random guessing gives 50%.** That is our baseline. Any accuracy near 50% means the
  model has learned nothing.
- **Accuracy means what you think it means.** With balanced classes, "83% accurate"
  really does mean the model is right about 5 images out of 6, on both classes roughly
  equally.

**Why bother checking?** Imagine the data had been 95% cats and 5% dogs. A model that
ignores the image completely and always answers "cat" would score **95% accuracy** —
and be worthless. This is the most common way of fooling yourself with an image
classifier. The defences are: look at the class counts (this section), and look at
performance *per class* rather than one global number (Section 18, the confusion
matrix).

---
# 4. Image Transformations

Between the JPEG on disk and the numbers entering the network sits a **transform
pipeline**. Ours does three things, plus optional extras for training:

| Step | What it does | Why |
|---|---|---|
| `Resize((128, 128))` | forces one common size | batches need identical shapes (Section 5) |
| `ToTensor()` | image → tensor of numbers between 0 and 1 | PyTorch works with tensors (Section 6) |
| `Normalize(0.5, 0.5)` | shifts the values to roughly −1 … +1 | inputs centred on zero train more smoothly |

**About `Normalize`.** With `mean=0.5` and `std=0.5` the calculation is
`(x - 0.5) / 0.5`, so `0` becomes `-1` and `1` becomes `+1`. These round numbers make
it easy to undo later when we want to *look* at an image: `x * 0.5 + 0.5`.

### Two pipelines, not one

- **Training:** augmentation **+** preprocessing. Augmentation shows the network a
  slightly different version of each image every time, which makes it harder to simply
  memorise the training set.
- **Validation and test:** preprocessing **only**. Evaluation must give the same
  answer every time. If validation images were randomly flipped, the validation score
  would wobble for reasons that have nothing to do with the model.

We keep the augmentation mild: a horizontal flip (a mirrored dog is still a dog) and a
small rotation (photos are not perfectly level). Nothing more.

In [ ]:
IMG_SIZE = 128

# Training: augmentation, then preprocessing
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# Validation and test: preprocessing only, nothing random
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print(train_transform)

### See the augmentation happen

The same image, passed through `train_transform` six times. Two of the steps are
random, so one file gives six different results — that is the whole idea.

We need one more small helper first: `to_displayable` undoes the normalisation and
puts the colour channels back where matplotlib expects them, so that we can look at a
tensor as a picture.

In [ ]:
def to_displayable(tensor):
    """Turn a normalised C x H x W tensor back into something we can plot."""
    image = tensor * 0.5 + 0.5                 # undo Normalize -> back to 0..1
    image = image.permute(1, 2, 0)             # C x H x W  ->  H x W x C
    return image.numpy().clip(0, 1)

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

sample = Image.open(train_dog_paths[0]).convert("RGB")

augmented = []
for i in range(6):
    augmented.append(to_displayable(train_transform(sample)))

show_images(augmented, [f"train_transform #{i + 1}" for i in range(6)], columns=6, size=2.2)

# The evaluation pipeline gives the same result every time
show_images([to_displayable(eval_transform(sample)), to_displayable(eval_transform(sample))],
            ["eval_transform #1", "eval_transform #2 (identical)"],
            columns=2, size=2.8)

---
# 5. Why Resize?

A batch of images is a single block of numbers with one shape:
`[batch, channels, height, width]`. There is no such thing as a batch where image 3 is
500x375 and image 4 is 200x300. So every image has to be brought to the same size
before it can be stacked with the others.

**Which size?** A trade-off:

- Bigger → more detail kept, but slower training and more memory.
- Smaller → faster, but fine detail (a whisker, an eye) disappears.

`128 x 128` is a good compromise for a teaching notebook: still recognisable to a
human, small enough to train on a laptop.

**One thing resizing does not change: the label.** A squashed dog is still a dog. The
`Resize` step touches the pixels; it never touches the answer.

In [ ]:
original = Image.open(train_cat_paths[0]).convert("RGB")
resized = transforms.Resize((IMG_SIZE, IMG_SIZE))(original)

print("original :", original.size, "(width, height)")
print("resized  :", resized.size)

show_images([original, resized],
            [f"original {original.size[0]}x{original.size[1]}", f"resized {IMG_SIZE}x{IMG_SIZE}"],
            columns=2, size=4)

Look closely at the resized image: unless the original happened to be square, it is
**stretched or squashed**. `Resize((128, 128))` ignores the original proportions and
forces both sides.

The usual alternative is `Resize(128)` followed by `CenterCrop(128)`, which keeps the
proportions but cuts off the edges of the picture. Neither option is free: one
distorts, the other throws pixels away. We take the distortion, because a cat's face
is still recognisable when stretched, whereas cropping can cut the animal out of the
frame entirely.

> **Question:** which of the two would you choose for X-ray images, where the
> proportions of the ribcage carry medical meaning? Why?

---
# 6. From Image to Tensor

A colour image is a grid of pixels, and each pixel has three numbers: red, green and
blue. So an image is really a block of numbers with three dimensions — and PyTorch
wants that block arranged in a particular order. Let us watch the conversion happen.

In [ ]:
image = Image.open(train_cat_paths[1]).convert("RGB")
resized = transforms.Resize((IMG_SIZE, IMG_SIZE))(image)

# Step 1 - as a plain NumPy array
as_numpy = np.array(resized)
print("NumPy array")
print("  shape :", as_numpy.shape, "  (height, width, channels)  <- channel LAST")
print("  dtype :", as_numpy.dtype)
print("  values:", as_numpy.min(), "to", as_numpy.max())

# Step 2 - ToTensor
tensor = transforms.ToTensor()(resized)
print()
print("After ToTensor()")
print("  shape :", tuple(tensor.shape), "  (channels, height, width)  <- channel FIRST")
print("  dtype :", tensor.dtype)
print("  values:", round(tensor.min().item(), 3), "to", round(tensor.max().item(), 3))

# Step 3 - Normalize
normalized = transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])(tensor)
print()
print("After Normalize(0.5, 0.5)")
print("  shape :", tuple(normalized.shape))
print("  values:", round(normalized.min().item(), 3), "to", round(normalized.max().item(), 3))

### The shape transition, in words

```
H x W x C          (375, 500, 3)      NumPy / PIL / OpenCV  - channel last
      ↓  ToTensor()
C x H x W          (3, 128, 128)      PyTorch - channel first, values 0..1
      ↓  the DataLoader stacks 32 of them into a batch
N x C x H x W      (32, 3, 128, 128)  what the network actually receives
```

**Why does PyTorch put the channel first?** Because of how convolution works. A filter
looks at a small window across *all* the channels at once, so it is convenient for
each channel to sit in memory as its own complete 2D image. `C x H x W` gives exactly
that: three separate `128 x 128` grids. `H x W x C` mixes R, G and B together at every
pixel, which is the natural layout for *displaying* an image and the awkward one for
convolving it.

That is the only reason our `to_displayable` helper has that `permute` line in it: to
put the channels back at the end so matplotlib can show the picture.

### The three channels are three grayscale images

In [ ]:
red = tensor[0].numpy()
green = tensor[1].numpy()
blue = tensor[2].numpy()

show_images([to_displayable(tensor), red, green, blue],
            ["RGB image", "red channel", "green channel", "blue channel"],
            columns=4, size=3)

print("The whole tensor  :", tuple(tensor.shape))
print("One channel of it :", tuple(tensor[0].shape), " <- just a 2D grid of numbers")

---
## 6.1 Datasets and DataLoaders

`ImageFolder` turns our folder structure into a labelled dataset. It looks inside
`dataset/train/`, sorts the folder names alphabetically, and gives each one a number:

```
cats/  →  0
dogs/  →  1
```

That ordering is not just cosmetic. It fixes the meaning of the model's output for the
whole notebook: **the single number the network produces is evidence for class 1,
which is `dog`.** A high value means dog, a low value means cat.

The `DataLoader` then hands us the images in batches.

- **`shuffle=True` for training only.** Batches should not be all cats and then all
  dogs.
- **`batch_size=32`** — how many images the network looks at before each update.
- **`num_workers=0`** — how many extra processes load images in the background. On
  Windows and inside notebooks, 0 is the safe choice.

In [ ]:
BATCH_SIZE = 32

train_dataset = ImageFolder(DATA_ROOT / "train", transform=train_transform)
val_dataset = ImageFolder(DATA_ROOT / "val", transform=eval_transform)
test_dataset = ImageFolder(DATA_ROOT / "test", transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

CLASS_NAMES = train_dataset.classes           # ['cats', 'dogs']

print("class to number :", train_dataset.class_to_idx)
print("train images    :", len(train_dataset), "->", len(train_loader), "batches")
print("val images      :", len(val_dataset), "->", len(val_loader), "batches")
print("test images     :", len(test_dataset), "->", len(test_loader), "batches")

> ### TODO 1 — print the shape of one batch
>
> Before running the next cell, write down what you expect the shape of `batch_images`
> to be. You know the batch size, the number of colour channels and the image size.
> Then run the cell and check.
>
> Also predict: what will the labels look like?

In [ ]:
batch_images, batch_labels = next(iter(train_loader))

print("images :", tuple(batch_images.shape))
print("labels :", tuple(batch_labels.shape))
print()
print("first 16 labels :", batch_labels[:16].tolist())
print("cats in this batch :", (batch_labels == 0).sum().item())
print("dogs in this batch :", (batch_labels == 1).sum().item())

In [ ]:
# Look at the batch the network will actually see: resized, normalised, classes mixed
first_eight = [to_displayable(batch_images[i]) for i in range(8)]
labels_eight = [CLASS_NAMES[batch_labels[i]] for i in range(8)]

show_images(first_eight, labels_eight, columns=8, size=1.8)

---
# 7. Build the CNN

Here is the whole architecture in one picture:

```
Input          3 x 128 x 128
   ↓  Conv2d(3, 16, k=3, p=1)  →  ReLU  →  MaxPool(2)
Block 1       16 x  64 x  64
   ↓  Conv2d(16, 32, k=3, p=1) →  ReLU  →  MaxPool(2)
Block 2       32 x  32 x  32
   ↓  Conv2d(32, 64, k=3, p=1) →  ReLU  →  MaxPool(2)
Block 3       64 x  16 x  16
   ↓  Flatten
Vector        16384
   ↓  Linear → ReLU → Dropout → Linear
Output        1 number
```

Three blocks, all built the same way: **convolution → activation → downsampling**.
That repeating pattern is the basic building block of almost every CNN. After the
three blocks, a small ordinary neural network turns the result into one number.

Two choices worth naming now:

- **`padding=1` with `kernel_size=3`** keeps the height and width unchanged through
  the convolution. All the shrinking is done by the pooling layers, one half at a
  time. That makes the arithmetic easy to follow, which is exactly why we do it this
  way.
- **Channels go up (3 → 16 → 32 → 64) while height and width go down (128 → 64 → 32 →
  16).** Standard CNN trade: give up precision about *where* things are, gain a richer
  description of *what* is there.

The model is written in two parts. `self.features` is the part that looks at the image;
`self.classifier` is the part that makes the decision. Splitting them is not required —
it just makes the model easier to talk about, and lets us peek inside later.

In [ ]:
class SimpleCNN(nn.Module):
    """A small CNN for cat vs dog. Three convolution blocks, then a classifier."""

    def __init__(self):
        super().__init__()

        # The part that looks at the image
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # The part that makes the decision.
        # 64 * 16 * 16 = 16384 is the number of values coming out of the blocks above.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),           # ONE output: the score for "dog"
        )

    def forward(self, x):
        x = self.features(x)             # 3 x 128 x 128  ->  64 x 16 x 16
        x = self.classifier(x)           # 64 x 16 x 16   ->  1 number
        return x


torch.manual_seed(SEED)
model = SimpleCNN().to(device)
print(model)

---
# 8. Understand the CNN Architecture

Now slow down and read one line properly:

```python
nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
```

| Argument | Value | Meaning |
|---|---|---|
| `in_channels` | 3 | how many channels the **input** has — here red, green, blue |
| `out_channels` | 16 | how many **filters** this layer has, and therefore how many result images it produces |
| `kernel_size` | 3 | each filter is a 3x3 window |
| `stride` | 1 | the window moves one pixel at a time |
| `padding` | 1 | one ring of zeros around the border, so the output keeps the same height and width |

### The thing everybody mixes up at first

> **`in_channels` is not the number of filters.**

`Conv2d(3, 16, 3)` means:

- the input has **3 channels**,
- the layer has **16 filters**,
- each filter is **3 x 3 x 3** — a 3x3 window looking at all 3 input channels at once,
  not a flat square,
- the output has **16 channels**, one result image per filter.

So one filter holds `3 * 3 * 3 = 27` numbers, plus 1 bias = 28. The layer has
`16 * 28 = 448` numbers to learn. Let us check that against PyTorch instead of
trusting the arithmetic.

In [ ]:
first_conv = model.features[0]

print("layer        :", first_conv)
print("weight shape :", tuple(first_conv.weight.shape))
print("               (number of filters, input channels, height, width)")
print("bias shape   :", tuple(first_conv.bias.shape))
print()
print("one filter   :", tuple(first_conv.weight[0].shape), " <- 3x3 across all 3 input channels")
print("numbers to learn in this layer :",
      first_conv.weight.numel() + first_conv.bias.numel())

### Questions — answer these before moving on

Looking at `Conv2d(3, 16, 3, padding=1)`:

1. What does the first `3` mean?
2. What does `16` mean?
3. What does the third argument, `3`, describe?
4. How many result images (feature maps) come out of this layer?
5. In the **second** block, `Conv2d(16, 32, 3, padding=1)`, what shape is one filter?
   How many numbers does that layer learn? Check your answer with the cell below.

In [ ]:
total = 0

for name, layer in model.named_modules():
    if isinstance(layer, nn.Conv2d) or isinstance(layer, nn.Linear):
        n_params = layer.weight.numel() + layer.bias.numel()
        total += n_params
        print(f"{name:<15}{str(tuple(layer.weight.shape)):<22}{n_params:>12,} numbers")

print()
print(f"{'TOTAL':<15}{'':<22}{total:>12,} numbers to learn")

### Read that table again

Look at where the numbers actually are. The three convolution layers together hold a
few thousand values. The first `Linear` layer holds **over two million**, because it
connects all 16,384 flattened values to 128 neurons.

That contrast is the argument for convolution:

- A filter is **small** (3x3) and is **used at every position** in the image. The same
  27 numbers examine the top-left corner and the bottom-right corner. Finding the same
  pattern somewhere else costs nothing extra — which is why a CNN copes with a cat
  that moved a bit.
- A fully connected layer has a **separate weight for every position**. Move the cat
  two pixels and every weight is looking at something different.

That is also why we shrink the image down to 16x16 before flattening. Feeding the
original 3 x 128 x 128 image straight into a Linear layer would need 49,152 inputs and
about 6 million weights in the first layer alone.

---
# 9. Track Tensor Shapes

**This is the most important section in the notebook.** If you can work out these
shapes by hand, you understand what a CNN does to an image.

The rule for one convolution or pooling layer, applied to height and to width:

```
output = (input + 2*padding - kernel_size) / stride  + 1
```

Apply it to our two kinds of layer:

- `Conv2d(kernel_size=3, stride=1, padding=1)`: `(input + 2 - 3) / 1 + 1 = input`.
  **Size unchanged.**
- `MaxPool2d(kernel_size=2, stride=2)`: `(input - 2) / 2 + 1 = input / 2`.
  **Size halved.**

The number of channels is not part of that formula. For a convolution it is simply
`out_channels`; pooling leaves it alone.

> ### TODO 2 — fill in this table before running anything
>
> The input is `3 x 128 x 128`. Work down the network and write the shape after each
> layer. Cover the output of the next cell while you do it.
>
> | after | channels | height | width |
> |---|---|---|---|
> | input         | 3 | 128 | 128 |
> | Conv2d(3,16)  | ? | ? | ? |
> | ReLU          | ? | ? | ? |
> | MaxPool(2)    | ? | ? | ? |
> | Conv2d(16,32) | ? | ? | ? |
> | ReLU          | ? | ? | ? |
> | MaxPool(2)    | ? | ? | ? |
> | Conv2d(32,64) | ? | ? | ? |
> | ReLU          | ? | ? | ? |
> | MaxPool(2)    | ? | ? | ? |
> | Flatten       | — | — | ? |
>
> Two extra questions:
> - Which layers change the **number of channels**? Which change the **height and
>   width**?
> - What does **ReLU** do to the shape? (Careful, this one is a trick question.)

In [ ]:
# Take one image from our batch and push it through the layers one at a time
x = batch_images[:1].to(device)      # keep it as a batch of 1

model.eval()
print("input          ", tuple(x.shape))

with torch.no_grad():
    for layer in model.features:
        x = layer(x)
        print(f"{layer.__class__.__name__:<15}", tuple(x.shape))

    flattened = x.flatten(1)
    print(f"{'Flatten':<15}", tuple(flattened.shape))

    output = model(batch_images[:1].to(device))
    print(f"{'Classifier':<15}", tuple(output.shape))

### What just happened to the image

- The **first number** in every shape is the batch size (1 here). Every layer works on
  a whole batch at once.
- **ReLU never changes the shape.** It just replaces every negative value with 0 and
  leaves positives alone. Same shape in, same shape out. (If you got that one wrong,
  you are in good company.)
- **Convolution changes the number of channels. MaxPool changes the height and
  width.** They are cleanly separated here because we used `padding=1`.
- The image started as `3 * 128 * 128 = 49,152` numbers and arrives at the classifier
  as `64 * 16 * 16 = 16,384`. We traded "exactly which pixel was bright" for "which
  patterns are present, roughly where" — and for deciding cat vs dog, the second is
  more useful.

> **Check yourself:** if the images were 160x160 instead, what shape would come out of
> the last MaxPool? (Answer: `64 x 20 x 20`.) What would you have to change in
> `SimpleCNN` for the model to still work?

---
# 10. Feature Maps

A **filter** and a **feature map** are different things, and the difference is the
central idea of this notebook:

| | what it is | shape here | how many |
|---|---|---|---|
| **Filter** (kernel) | the learned numbers — *the thing that looks* | 3 x 3 x 3 | 16 in block 1 |
| **Feature map** | the result of sliding one filter over the image — *what it saw* | 128 x 128 | 16, one per filter |

Each filter slides over the input and computes a multiply-and-add at every position.
The grid of results is that filter's feature map: **bright where the filter's pattern
was found, dark where it was not**.

Right now our model is **untrained**, so its filters are still random numbers and the
feature maps will not look like much. Look at them anyway — this is the "before"
picture, and we will run the same code again after training.

Two small helpers: one to get the feature maps out of a chosen layer, one to draw
them.

In [ ]:
def get_feature_maps(model, image, layer_index):
    """Push one image through model.features up to `layer_index` and return the result."""
    x = image.unsqueeze(0).to(device)         # add the batch dimension: C,H,W -> 1,C,H,W

    model.eval()
    with torch.no_grad():
        for layer in model.features[:layer_index + 1]:
            x = layer(x)

    return x[0].cpu()                         # remove the batch dimension again


def show_feature_maps(maps, title, how_many=16):
    """Draw the first `how_many` feature maps as a grid of small images."""
    how_many = min(how_many, len(maps))
    print(title, "- shape of this layer's output:", tuple(maps.shape))

    plt.figure(figsize=(12, 1.6 * np.ceil(how_many / 8)))
    for i in range(how_many):
        plt.subplot(int(np.ceil(how_many / 8)), 8, i + 1)
        plt.imshow(maps[i], cmap="viridis")
        plt.title(f"map {i}", fontsize=7)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# One image that we will keep coming back to
demo_image, demo_label = test_dataset[3]
show_images([to_displayable(demo_image)], [f"input: {CLASS_NAMES[demo_label]}"], columns=1, size=3)

# Layer 0 is the first Conv2d
maps = get_feature_maps(model, demo_image, layer_index=0)
show_feature_maps(maps, "UNTRAINED: after the first Conv2d")

The layers of `model.features` are numbered in the order you wrote them:

```
0  Conv2d(3, 16)      3  Conv2d(16, 32)     6  Conv2d(32, 64)
1  ReLU               4  ReLU               7  ReLU
2  MaxPool2d          5  MaxPool2d          8  MaxPool2d
```

> ### TODO 4 — explore the feature maps
>
> 1. Run `get_feature_maps(model, demo_image, layer_index=1)` — that is the **ReLU**
>    right after the first convolution. Compare it with layer 0. What happened to the
>    dark areas, and why? (Hint: what does ReLU do to negative numbers?)
> 2. Try `layer_index=2`, the first MaxPool. What changed about the *size*? Can you
>    still recognise the animal?
> 3. Try `layer_index=6`, the third convolution. Can you still see an animal at all?
> 4. Try a different image (`test_dataset[10]`, `test_dataset[25]`, ...).

In [ ]:
# Your experiments here
maps = get_feature_maps(model, demo_image, layer_index=1)
show_feature_maps(maps, "UNTRAINED: after the first ReLU")

### How to talk about feature maps honestly

It is tempting to look at a map and announce "filter 4 is an edge detector". Resist
that. What we can safely say is:

> The network learns filters that respond to patterns **useful for this task**. A
> feature map shows how strongly one filter responded at each position.

Some filters in the first layer do end up looking edge-like or colour-like, which
resembles the filters people used to design by hand. But nobody told the network to
learn those, nobody labelled them, and the deeper layers respond to combinations that
have no simple name.

That is the shift worth remembering: **a hand-designed filter has numbers a person
chose; a CNN filter has numbers the data chose.**

---
# 11. Build the Classifier

The blocks end with `64 x 16 x 16`: 64 feature maps, each a 16x16 grid. That is a
*description* of the image, not a decision. The classifier turns it into one.

```
64 x 16 x 16 feature maps
      ↓  Flatten            lay the whole block out as one long list
16384 numbers
      ↓  Linear(16384, 128) + ReLU + Dropout
128 numbers                 a short summary of the image
      ↓  Linear(128, 1)
1 number (a "logit")        the score for "dog"
      ↓  sigmoid            (only when we want to read it)
a probability between 0 and 1
```

**Why one output and not two?** With two classes, one number is enough: if the
probability of dog is 0.9, then the probability of cat is 0.1. A second output would
be redundant.

**Why no sigmoid inside the model?** Because the loss function in the next section
applies it internally, in a more numerically stable way. The model gives a raw score;
we apply `sigmoid` ourselves only when we want a readable probability.

- score `0` → probability `0.5` — completely unsure
- score above 0 → probability above 0.5 → **dog**
- score below 0 → probability below 0.5 → **cat**

**`Dropout(0.3)`** randomly switches off 30% of those 128 numbers during training, so
the model cannot lean too heavily on any single one. It turns itself off when we call
`model.eval()`.

In [ ]:
model.eval()
with torch.no_grad():
    scores = model(batch_images[:6].to(device))     # shape [6, 1]
    probabilities = torch.sigmoid(scores)

print("true      score    P(dog)   prediction")
print("-" * 42)
for i in range(6):
    score = scores[i].item()
    probability = probabilities[i].item()
    prediction = CLASS_NAMES[1] if probability > 0.5 else CLASS_NAMES[0]
    print(f"{CLASS_NAMES[batch_labels[i]]:<10}{score:>6.2f}{probability:>10.2f}   {prediction}")

print()
print("The model is untrained, so these answers mean nothing -")
print("but the plumbing works: image in, one probability out.")

---
# 12. Loss Function

We use `nn.BCEWithLogitsLoss` — binary cross entropy, applied to the raw score.

What it does, in one sentence: **it measures how far the predicted probability is from
the true answer (0 for cat, 1 for dog), and punishes confident mistakes much harder
than hesitant ones.**

Two practical consequences for our code:

- The model must output a **raw score**, with no sigmoid. (Ours does.)
- The targets must be **floats with the same shape as the output**, not the whole
  numbers the DataLoader gives us. That is why you will see
  `labels.float().unsqueeze(1)` in the training loop: it turns a list of 32 labels
  into a column of 32 floats.

The numbers below show the shape of the punishment. Look at the last row.

In [ ]:
criterion = nn.BCEWithLogitsLoss()

examples = [
    ("confident and RIGHT", 4.0),
    ("hesitant and right ", 0.5),
    ("no idea            ", 0.0),
    ("hesitant and wrong ", -0.5),
    ("confident and WRONG", -4.0),
]

print("The true answer is 'dog' (target = 1) in every row.")
print()
print("case                  score   P(dog)    loss")
print("-" * 46)

for description, score in examples:
    score_tensor = torch.tensor([[score]])
    target_tensor = torch.tensor([[1.0]])

    probability = torch.sigmoid(score_tensor).item()
    loss = criterion(score_tensor, target_tensor).item()

    print(f"{description}{score:>8.1f}{probability:>9.2f}{loss:>8.3f}")

Being confidently wrong costs roughly eight times what being merely unsure costs. That
is what pushes the model towards honest confidence instead of bluffing.

> **A number worth remembering:** a model that answers 0.5 for everything has a loss
> of about **0.69**. If your training loss sits at 0.69 and will not move, the model is
> not learning anything.

---
# 13. Optimizer

```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
```

`model.parameters()` is every number the model can learn: the biases, the weights of
the Linear layers, **and the convolution filters themselves**. That last part is worth
pausing on — the optimizer adjusting "weights" *is* the mechanism by which the filters
are discovered. Filters are not special; they are just numbers that happen to be used
as a sliding window.

`1e-3` (0.001) is the usual starting point for Adam. Too high and the loss bounces
around; too low and training crawls. It is the first thing worth changing in Section
22.

In [ ]:
LEARNING_RATE = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

n_numbers = sum(p.numel() for p in model.parameters())
print("The optimizer will be adjusting", f"{n_numbers:,}", "numbers.")
print("The first group of them is the filter bank of block 1:",
      tuple(model.features[0].weight.shape))

---
# 14. Training Loop

No framework, no `.fit()`. One function, ten steps, all visible. Read it once before
running it.

```
1. model.train()                 put the model in training mode
2. for images, labels in loader  go through the data in batches
3. .to(device)                   move the data to where the model is
4. outputs = model(images)       FORWARD pass
5. loss = criterion(...)         how wrong was it?
6. optimizer.zero_grad()         clear the gradients from last time
7. loss.backward()               BACKWARD pass - work out the gradients
8. optimizer.step()              update every number in the model
9. total_loss += ...             keep track of the loss
10. correct += ...               keep track of the accuracy
```

Two things that catch people out:

- **`optimizer.zero_grad()` is not optional.** PyTorch *adds up* gradients by default.
  Forget this line and batch 5's update is based on batches 1 to 5 all mixed together.
- **`loss.item()`** takes the plain number out of the tensor. Adding up the tensors
  themselves would slowly eat your memory.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Go through the training set once, learning as we go. Returns loss and accuracy."""
    model.train()                                   # 1. training mode (dropout is ON)

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:                   # 2. one batch at a time
        images = images.to(device)                  # 3. move to the device
        targets = labels.float().unsqueeze(1).to(device)

        outputs = model(images)                     # 4. forward pass
        loss = criterion(outputs, targets)          # 5. how wrong?

        optimizer.zero_grad()                       # 6. clear old gradients
        loss.backward()                             # 7. backpropagation
        optimizer.step()                            # 8. update the model

        total_loss += loss.item()                   # 9. track the loss

        predictions = (torch.sigmoid(outputs) > 0.5).float()    # 10. track accuracy
        correct += (predictions == targets).sum().item()
        total += len(labels)

    return total_loss / len(loader), correct / total

---
# 15. Validation Loop

Almost the same loop, with three deliberate differences:

- **`model.eval()`** puts the model in evaluation mode. For us that means **dropout is
  switched off**.
- **`torch.no_grad()`** tells PyTorch not to prepare for learning. We are not going to
  call `backward()` here, so this saves time and memory.
- **No `loss.backward()`, no `optimizer.step()`.** Nothing is updated.

**Why must validation not update the model?** Because the validation set's job is to
give an *honest* estimate of how the model does on images it has not learned from. The
moment a validation image changes a weight, it is no longer unseen, and the number it
produces stops meaning anything.

We look at the validation score after every epoch, to see how training is going. The
**test** set is stricter: we look at it once, at the very end, in Section 17.

In [ ]:
def evaluate(model, loader, criterion):
    """Go through a dataset without learning. Returns loss and accuracy."""
    model.eval()                                    # evaluation mode (dropout is OFF)

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():                           # no learning happens in here
        for images, labels in loader:
            images = images.to(device)
            targets = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            total_loss += loss.item()

            predictions = (torch.sigmoid(outputs) > 0.5).float()
            correct += (predictions == targets).sum().item()
            total += len(labels)

    return total_loss / len(loader), correct / total

## 15.1 Run the training

Now put the two loops together. Each epoch: train on all 2,000 training images, then
check on the 400 validation images, then write down four numbers.

Expect a few minutes on a CPU, well under a minute on a GPU. If it is too slow, lower
`EPOCHS`.

In [ ]:
import time

EPOCHS = 10

# Start from a fresh model, so this cell gives the same result every time you run it
torch.manual_seed(SEED)
model = SimpleCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print("epoch   train_loss  train_acc   val_loss   val_acc    time")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    started = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"{epoch:^7}{train_loss:>10.3f}{train_acc:>11.3f}"
          f"{val_loss:>11.3f}{val_acc:>10.3f}{time.time() - started:>8.1f}s")

print("-" * 60)
print("best validation accuracy:", round(max(history["val_acc"]), 3))

torch.save(model.state_dict(), "cnn_cat_dog.pt")
print("model saved to cnn_cat_dog.pt")

---
# 16. Plot the Training Curves

The table above is hard to read. The same four numbers drawn as curves are much
easier, and reading these plots is a real skill.

In [ ]:
epoch_numbers = range(1, EPOCHS + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epoch_numbers, history["train_loss"], "o-", label="train")
plt.plot(epoch_numbers, history["val_loss"], "s-", label="validation")
plt.axhline(0.69, color="gray", linestyle=":", label="random guessing")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Loss")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epoch_numbers, history["train_acc"], "o-", label="train")
plt.plot(epoch_numbers, history["val_acc"], "s-", label="validation")
plt.axhline(0.5, color="gray", linestyle=":", label="random guessing")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Accuracy")
plt.ylim(0.4, 1.02)
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

gap = history["train_acc"][-1] - history["val_acc"][-1]
print("final train accuracy :", round(history["train_acc"][-1], 3))
print("final val accuracy   :", round(history["val_acc"][-1], 3))
print("gap between them     :", round(gap, 3))

### Read your own curves — answer in writing

1. **Is the model learning at all?** The training loss should drop clearly below 0.69
   and the accuracy should climb above 0.5. If not, nothing else here matters.
2. **Is the validation score still improving,** or has it flattened out?
3. **Is it overfitting?** Look at the *gap* between the two lines. Overfitting looks
   like this:
   - training loss keeps falling, but **validation loss turns around and rises**;
   - training accuracy heads for 0.99 while validation accuracy stalls;
   - the two curves spread apart instead of staying together.

   It means the model is memorising the training images instead of learning what a cat
   looks like. With 2,000 images and a model this size, expect to see some of it.
4. **Which epoch would you have stopped at?** Training past the best validation score
   does not help. Stopping there is called **early stopping**, and a real project would
   save the model from that epoch rather than the last one.
5. **Is the validation curve jumpy?** With only 400 validation images, one image is
   0.25% of the accuracy. Small jumps between epochs are noise, not information.

**Ways to reduce the gap:** more data, more augmentation, more dropout, a smaller
model, or stopping earlier.

## 16.1 Feature maps, now that the filters are trained

Same image and same code as Section 10 — but the filters have learned something since
then. Compare the two figures.

In [ ]:
maps = get_feature_maps(model, demo_image, layer_index=0)
show_feature_maps(maps, "TRAINED: after the first Conv2d")

maps = get_feature_maps(model, demo_image, layer_index=6)
show_feature_maps(maps, "TRAINED: after the third Conv2d (first 16 of 64 maps)")

What to look for, without over-claiming:

- The **first-layer** maps are more structured than they were before training. Some
  now pick out outlines, textures, or particular colours.
- The **third-layer** maps are small (32x32) and abstract. You may not see an animal at
  all. Those units respond to combinations of earlier features over a much bigger area
  of the original photo — the area a unit can "see" is called its **receptive field** —
  and there is no reason for those combinations to have a name.
- Some maps are almost entirely dark. That is normal: those filters simply did not find
  their pattern in this particular image.

---
# 17. Evaluate on the Test Set

The test set has been untouched until now. No gradient came from it, and no decision
about epochs or architecture was based on it. That is what makes this number an
estimate of how the model does on genuinely new photos.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)

n_test = len(test_dataset)
n_correct = round(test_acc * n_test)

print("test loss     :", round(test_loss, 3))
print("test accuracy :", round(test_acc, 3))
print("correct       :", n_correct, "out of", n_test)
print("incorrect     :", n_test - n_correct, "out of", n_test)
print()
print("for comparison: random guessing = 0.5")

### Why one number is not enough

Suppose that says 82%. Useful, but it hides everything you would actually want to
know:

- Are the mistakes **split evenly** between cats and dogs, or is the model much worse
  at one of them? → the confusion matrix, Section 18.
- Are the mistakes **confident** or **borderline**? A wrong answer at 0.51 is a coin
  flip; a wrong answer at 0.99 is a model that is sure and wrong. → Section 19.
- **What kind** of images does it fail on? → Section 20.

Accuracy is a summary. The interesting work starts when you ask what it is summarising.

## 17.1 Collect the predictions

Everything below needs the model's answer for each individual test image, so we
collect them all once. Because `test_loader` was built with `shuffle=False`, the
answers come back in the same order as the dataset: answer number `i` belongs to
`test_dataset[i]`.

In [ ]:
model.eval()

all_probabilities = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        probabilities = torch.sigmoid(outputs)

        all_probabilities.extend(probabilities.cpu().flatten().tolist())
        all_labels.extend(labels.tolist())

probs = np.array(all_probabilities)      # P(dog) for each test image
y_true = np.array(all_labels)            # 0 = cat, 1 = dog
preds = (probs > 0.5).astype(int)        # what the model decided

print("collected", len(probs), "predictions")
print("accuracy from these arrays:", round((preds == y_true).mean(), 3))

plt.figure(figsize=(7, 3))
plt.hist(probs[y_true == 0], bins=25, alpha=0.7, label="true cats")
plt.hist(probs[y_true == 1], bins=25, alpha=0.7, label="true dogs")
plt.axvline(0.5, color="black", linestyle="--", label="decision line")
plt.xlabel("P(dog)")
plt.ylabel("number of images")
plt.title("How confident is the model?")
plt.legend()
plt.tight_layout()
plt.show()

That histogram is worth a moment. Ideally the cats pile up near 0 and the dogs near 1,
with few images sitting near the 0.5 line. Images in the middle are the ones the model
finds genuinely unclear; images far on the *wrong* side are the interesting failures.

---
# 18. Confusion Matrix

Accuracy squashes four different numbers into one. The confusion matrix keeps them
apart. Because `dogs` is class 1, we call dog the "positive" answer:

|  | predicted **cat** | predicted **dog** |
|---|---|---|
| **actually cat** | True Negative (TN) — correct | False Positive (FP) — a cat called a dog |
| **actually dog** | False Negative (FN) — a dog called a cat | True Positive (TP) — correct |

The diagonal is what the model got right. Everything off the diagonal is a mistake,
and the two kinds of mistake are not always equally bad — in a medical setting, one of
them can be far more expensive than the other.

We count them with a simple loop, because writing it once makes the definitions stick.

In [ ]:
true_negative = 0     # cat, called cat
false_positive = 0    # cat, called dog
false_negative = 0    # dog, called cat
true_positive = 0     # dog, called dog

for true_label, predicted_label in zip(y_true, preds):
    if true_label == 0 and predicted_label == 0:
        true_negative += 1
    elif true_label == 0 and predicted_label == 1:
        false_positive += 1
    elif true_label == 1 and predicted_label == 0:
        false_negative += 1
    else:
        true_positive += 1

print("True  Negative (cat -> cat) :", true_negative)
print("False Positive (cat -> dog) :", false_positive)
print("False Negative (dog -> cat) :", false_negative)
print("True  Positive (dog -> dog) :", true_positive)
print()
print("accuracy on cats :", round(true_negative / (true_negative + false_positive), 3))
print("accuracy on dogs :", round(true_positive / (true_positive + false_negative), 3))

matrix = np.array([[true_negative, false_positive],
                   [false_negative, true_positive]])

plt.figure(figsize=(4.5, 4))
plt.imshow(matrix, cmap="Blues")
plt.xticks([0, 1], ["predicted cat", "predicted dog"])
plt.yticks([0, 1], ["actual cat", "actual dog"])
for row in range(2):
    for column in range(2):
        plt.text(column, row, matrix[row, column], ha="center", va="center", fontsize=18)
plt.title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

> **Look at your matrix and answer:**
>
> - Are the two mistake cells roughly equal, or does the model lean towards one class?
> - If this classifier decided which animals a shelter's app lists as dogs, which
>   mistake would you rather have? Does your answer change if the two classes were
>   "healthy" and "tumour"?
> - What would the confusion matrix of a model that always answers "dog" look like?
>   What accuracy would it get on **our** test set — and what would it get if the test
>   set were 95% dogs?

---
# 19. Visualize Predictions

Numbers are abstract. Look at the actual images together with what the model said.
Green means correct, red means wrong. The probability shown is always `P(dog)`.

In [ ]:
def show_predictions(indices, title):
    """Show test images with the true label, the prediction and the probability."""
    plt.figure(figsize=(15, 3 * int(np.ceil(len(indices) / 6))))

    for position, index in enumerate(indices):
        image, true_label = test_dataset[index]
        probability = probs[index]
        predicted_label = preds[index]
        is_correct = (predicted_label == true_label)

        plt.subplot(int(np.ceil(len(indices) / 6)), 6, position + 1)
        plt.imshow(to_displayable(image))
        plt.title(f"actual: {CLASS_NAMES[true_label]}\n"
                  f"predicted: {CLASS_NAMES[predicted_label]}\n"
                  f"P(dog) = {probability:.2f}",
                  fontsize=9,
                  color="green" if is_correct else "red")
        plt.axis("off")

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
random.seed(SEED)
twelve_random = random.sample(range(len(test_dataset)), 12)
show_predictions(twelve_random, "A random dozen test images")

In [ ]:
# Split the test images into the ones we got right and the ones we got wrong
correct_indices = [i for i in range(len(probs)) if preds[i] == y_true[i]]
wrong_indices = [i for i in range(len(probs)) if preds[i] != y_true[i]]

print(len(correct_indices), "correct,", len(wrong_indices), "wrong")

# How sure was the model about the answer it gave? (0.5 = unsure, 1.0 = completely sure)
confidence = np.maximum(probs, 1 - probs)

# The ones it was most sure about and right
most_sure_correct = sorted(correct_indices, key=lambda i: -confidence[i])[:6]
show_predictions(most_sure_correct, "Sure, and correct")

# The ones it hesitated over the most
least_sure = sorted(range(len(probs)), key=lambda i: confidence[i])[:6]
show_predictions(least_sure, "Least sure predictions (closest to the 0.5 line)")

> **Question about the unsure ones:** what do those images have in common? Look for the
> animal being small in the frame, partly hidden, in a strange pose, or sharing the
> photo with something else. Would *you* have hesitated?

---
# 20. Investigate Failure Cases

This is the part that separates running a model from engineering one. A model's
mistakes are information: they tell you something about the model, about the dataset,
and sometimes about the labels.

Start with the worst ones — the images the model got wrong **while being most sure**.

In [ ]:
most_sure_wrong = sorted(wrong_indices, key=lambda i: -confidence[i])[:12]
show_predictions(most_sure_wrong, "Sure, and WRONG - the model's worst mistakes")

> ### TODO 6 — investigate a misclassified image
>
> Pick one image from the grid above and look at it properly. Go through this list and
> write down which points apply:
>
> - **Unusual pose** — lying down, seen from behind, jumping?
> - **Something in the way** — a hand, a blanket, furniture, another animal?
> - **Lighting** — very dark, blown out by the flash, heavy shadow?
> - **Scale** — a tiny animal in a big room, or a close-up of one body part?
> - **Busy background** — hard to tell where the animal ends?
> - **Image quality** — blurry, low resolution, or a drawing rather than a photo?
> - **Genuinely unclear** — a small fluffy dog that honestly looks like a cat, or both
>   animals in one photo?
> - **Wrong label** — it happens. This dataset contains a few images that are not what
>   the folder says, and even a few with no animal in them at all.
>
> The cell below shows any test image at full size next to the squashed 128x128 version
> the model actually saw. Change `INDEX` to look at different ones.

In [ ]:
INDEX = most_sure_wrong[0]        # <- change this to any test image number

path, true_label = test_dataset.samples[INDEX]
original = Image.open(path).convert("RGB")
what_the_model_saw, _ = test_dataset[INDEX]

print("file       :", path)
print("size       :", original.size)
print("true label :", CLASS_NAMES[true_label])
print("predicted  :", CLASS_NAMES[preds[INDEX]], "with P(dog) =", round(float(probs[INDEX]), 3))

show_images([original, to_displayable(what_the_model_saw)],
            ["original image", f"what the model saw ({IMG_SIZE}x{IMG_SIZE})"],
            columns=2, size=4)

### What failures are worth to you

Sort what you found into three buckets, because each one suggests a different fix:

| Bucket | Example | What to do about it |
|---|---|---|
| **Model limitation** | fails on every dog seen from behind | a bigger model, longer training |
| **Data limitation** | almost no dark indoor photos in training | collect more of that kind, or augment for it |
| **Bad data** | a cartoon, or a photo with no animal | fix the label or remove the image |

And notice what you *cannot* do: you cannot look at the test set, notice the model
fails on dark images, tune the model until those pass, and still call the test accuracy
honest. Once you tune against it, it has quietly become validation data. That is why
real projects keep a test set locked away.

---
# 21. OPTIONAL — Data Augmentation Experiment

> These last two sections train more models, and each run takes about as long as the
> one in Section 15.1. Skip them if you are short on time and come back later.

We claimed in Section 4 that augmentation helps. Claims are cheap; let us test it.

**The setup matters.** To compare A and B fairly, exactly one thing may be different
between them. Same model, same learning rate, same number of epochs, same data — only
the training transform changes:

- **Experiment A** — resize, tensor, normalize. No augmentation.
- **Experiment B** — the same, plus flip and small rotation.

One function does a whole run, so that everything we are not comparing stays the same
automatically. It is the training loop from Section 15.1, wrapped up.

In [ ]:
def run_experiment(name, transform, epochs=8, learning_rate=1e-3):
    """Train a brand new model with the given training transform. Returns its history."""
    torch.manual_seed(SEED)                          # same starting point every time

    loader = DataLoader(ImageFolder(DATA_ROOT / "train", transform=transform),
                        batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    new_model = SimpleCNN().to(device)
    new_criterion = nn.BCEWithLogitsLoss()
    new_optimizer = torch.optim.Adam(new_model.parameters(), lr=learning_rate)

    result = {"name": name, "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(new_model, loader, new_criterion, new_optimizer)
        val_loss, val_acc = evaluate(new_model, val_loader, new_criterion)

        result["train_acc"].append(train_acc)
        result["val_acc"].append(val_acc)

        print(f"  [{name}] epoch {epoch:2d}   train_acc {train_acc:.3f}   val_acc {val_acc:.3f}")

    return result

In [ ]:
def compare(results):
    """Plot validation accuracy for several runs, and print a small summary."""
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    for result in results:
        plt.plot(range(1, len(result["val_acc"]) + 1), result["val_acc"], "o-", label=result["name"])
    plt.title("Validation accuracy")
    plt.xlabel("epoch")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    for result in results:
        gaps = [t - v for t, v in zip(result["train_acc"], result["val_acc"])]
        plt.plot(range(1, len(gaps) + 1), gaps, "o-", label=result["name"])
    plt.title("Overfitting gap (train accuracy - val accuracy)")
    plt.xlabel("epoch")
    plt.axhline(0, color="gray", linestyle=":")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    for result in results:
        best = max(result["val_acc"])
        final_gap = result["train_acc"][-1] - result["val_acc"][-1]
        print(f"{result['name']:<25} best val accuracy {best:.3f}   final gap {final_gap:.3f}")

> ### TODO 7 — compare training with and without augmentation
>
> Run the cell below (it trains two models, so give it time). But first, **write down
> your prediction**: which run will reach the higher validation accuracy, and which
> will have the bigger gap between training and validation?

In [ ]:
no_augmentation = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

with_augmentation = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

result_a = run_experiment("A: no augmentation", no_augmentation, epochs=8)
result_b = run_experiment("B: flip + rotation", with_augmentation, epochs=8)

compare([result_a, result_b])

### Interpreting the comparison

**Did augmentation help?** Compare the best validation accuracy. Then compare the gap:
augmentation's most reliable effect is a *smaller* gap, because the model never sees
exactly the same image twice and therefore has less to memorise. It is quite possible
for run B to have the smaller gap and *not* the higher accuracy, especially over only
8 epochs.

**Why might it help?** Because it tells the model something we already know about the
world: a mirrored dog is still a dog, and a photo tilted by 8 degrees is the same
scene. Rather than hoping our 1,000 images happen to cover every angle, we create the
variation ourselves.

**When does it not help?**

- When the change destroys the label. Flipping a photo of the digit "2" does not give
  you a "2". Cats and dogs survive mirroring; not everything does.
- When it is too aggressive — upside-down images and wild colour changes give you
  training images that look nothing like the test images.
- When the model is underfitting rather than overfitting. Augmentation makes training
  *harder*; if the model is already struggling, it makes things worse.

Do not write down "augmentation improves accuracy" as a rule. Write down what your run
showed, with the settings and the number of epochs. **One run is one data point** —
with a different seed, two close results can swap places.

---
# 22. OPTIONAL — Model Experiments

The skill being practised here is **change one thing at a time**. Change three things,
get a better number, and you have learned nothing about which change mattered.

Keep a table as you go:

| # | What I changed | What I expected | What actually happened | What I conclude |
|---|---|---|---|---|
| baseline | — | — | | |
| 1 | | | | |
| 2 | | | | |

> ### TODO 3 — change the number of filters from 16 to 32
>
> Scroll back to `SimpleCNN` in Section 7 and change the three convolution layers from
> `16, 32, 64` filters to `32, 64, 128`.
>
> **One other line has to change with it.** The classifier starts with
> `nn.Linear(64 * 16 * 16, 128)`, and that `64` was the number of filters in the last
> block. What does it become? (If you get it wrong, PyTorch will tell you — read the
> error message, it names both shapes.)
>
> Then rerun the model cell and the training cell in Section 15.1.
>
> Predict first: more filters means more capacity. Will that give better validation
> accuracy, a bigger overfitting gap, or both?

> ### TODO 5 — modify the CNN architecture
>
> Now change something else, one at a time. Pick one, run it, write down the result,
> then move on to the next.
>
> | Change | How | What to watch |
> |---|---|---|
> | Learning rate | `run_experiment(..., learning_rate=1e-2)` or `1e-4` | too high: the loss bounces or sticks near 0.69. too low: improvement crawls |
> | Kernel size | in `SimpleCNN`, use `kernel_size=5, padding=2` | each filter sees a bigger area, and has more numbers to learn |
> | No dropout | change `nn.Dropout(0.3)` to `nn.Dropout(0.0)` | the gap should get wider — that is dropout doing its job |
> | A fourth block | add `Conv2d(64, 128, 3, padding=1)`, `ReLU()`, `MaxPool2d(2)` | the maps become `128 x 8 x 8`, so the Linear layer becomes `128 * 8 * 8` |
> | Smaller images | set `IMG_SIZE = 64` and rerun from Section 4 | epochs get much faster; the Linear layer becomes `64 * 8 * 8`. Is the accuracy much worse? |
> | Stronger augmentation | add `transforms.ColorJitter(0.3, 0.3, 0.3)` | does more augmentation keep helping, or does it start to hurt? |
>
> Notice how many of these force you to update the `Linear` layer. That is Section 9
> coming back: if you know the shapes, you know what to change.

In [ ]:
# Your experiment. Change one thing, keep everything else the same.
my_result = run_experiment("my run: lr = 1e-4", with_augmentation, epochs=8, learning_rate=1e-4)

compare([result_b, my_result])

---
# Final Reflection

Answer these in your own words, without scrolling back up first. If you get stuck on
one, that is exactly the section to reread.

1. **Why are CNNs suitable for image data?** (Think about a small filter being reused
   at every position, and about what happens when the animal moves a few pixels.)
2. **What does a convolution filter learn?**
3. **What is the difference between a filter and a feature map?**
4. In `Conv2d(3, 16, 3)`, **what does each number mean?**
5. **Why does the number of channels grow deeper in the network?**
6. **Why do the height and width usually shrink?**
7. **What does pooling do, and why do we want it?**
8. **What is a receptive field?** Why does a unit in block 3 see a bigger part of the
   original photo than a unit in block 1, even though every filter is only 3x3?
9. **Why do we use `BCEWithLogitsLoss` here?** Why does the model output a raw score
   instead of a probability?
10. **Why do we split the data into training, validation and test?** What goes wrong if
    you tune your model against the test set?
11. **What does overfitting look like in the training curves?** Name two things you
    could do about it.
12. **Why does the model get some images right and others wrong?** Give one reason
    about the model and one about the data.

---
# The Conceptual Map

Everything in this notebook, in one column:

```
Image                    a JPEG on disk, any size
  ↓
Resize / Transform       128 x 128, tensor, normalized
  ↓
Tensor                   3 x 128 x 128     (channels, height, width)
  ↓
Conv Layer               16 filters slide across the input
  ↓
Learned Filters          3 x 3 x 3 numbers each - found during training
  ↓
Feature Maps             16 x 128 x 128    "where did each filter respond?"
  ↓
Activation (ReLU)        16 x 128 x 128    negatives become zero
  ↓
Downsampling (MaxPool)   16 x 64 x 64      half the height, half the width
  ↓
More Learned Features    32 x 32 x 32  →  64 x 16 x 16
  ↓
Feature Representation   16384 numbers describing the image
  ↓
Classifier               Linear → ReLU → Dropout → Linear
  ↓
One score → sigmoid      P(dog) = 0.91
  ↓
Cat / Dog                "Dog"
```

### The one idea to take away

A convolution filter is a small window of numbers that slides over an image, and the
result tells you where that pattern appears. That operation is old, and for decades
people chose the numbers by hand: a particular 3x3 grid finds edges, another one blurs.

A CNN performs the exact same operation. The only difference is where the numbers come
from:

| | filter chosen by a person | filter learned by a CNN |
|---|---|---|
| Where the numbers come from | designed by hand | found by training on labelled data |
| What it is good at | what the designer had in mind | whatever helps with the task |
| How many | one or two, applied one after another | thousands, stacked in layers |
| The operation itself | slide, multiply, add | slide, multiply, add — *identical* |
| Can you explain it? | completely — you wrote the numbers | partly — early layers look familiar, deep ones do not |

> **A hand-designed filter: a person chooses the numbers.**
> **A CNN filter: the data chooses the numbers.**

You now have a working classifier, a training and validation pipeline you wrote
yourself, an honest evaluation on unseen images, and a set of failure cases you have
actually looked at. That last one is the part most people skip.